# Rep-stat distribution explorer

Reads `data/derived/per_rep_stats.csv` (from `scripts/compute_rep_stats.py`)
and plots the distributions that drive threshold derivation in
`scripts/derive_thresholds.py`.

If the CSV is missing or empty, every cell short-circuits with a friendly
message — so you can open this notebook on a fresh clone and still read
the analysis plan end-to-end.

**Feeds into:** the P20/P75 percentiles this notebook plots are the exact
ones `derive_thresholds.py` consumes. Any change to percentile choice
should be prototyped here first.

In [ ]:
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().resolve().parents[0]
STATS_CSV = REPO_ROOT / 'data' / 'derived' / 'per_rep_stats.csv'

if not STATS_CSV.exists():
    print(f'No stats CSV at {STATS_CSV}. Run scripts/compute_rep_stats.py first.')
    df = pd.DataFrame()
else:
    df = pd.read_csv(STATS_CSV)

print(f'Loaded {len(df)} rep rows from {STATS_CSV.name}')
df.head()

## Quality-label breakdown

Only `good` reps feed the percentile math. Everything else is ignored — but
the class balance is worth eyeballing so we don't silently over-fit to a
tiny good-rep cohort.

In [ ]:
if df.empty:
    print('Skipping — no data.')
else:
    display(df['quality'].value_counts())

## Angle distributions for good reps

Histograms of `start_angle`, `peak_angle`, `end_angle` across good reps.
The red lines mark the percentiles that threshold derivation uses.

In [ ]:
import matplotlib.pyplot as plt

if df.empty:
    print('Skipping — no data.')
else:
    good = df[df['quality'] == 'good']
    if good.empty:
        print('No rows labelled "good" — nothing to plot.')
    else:
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        for ax, col, pct in zip(axes, ['start_angle', 'peak_angle', 'end_angle'], [20, 75, 20]):
            values = good[col].dropna()
            if values.empty:
                ax.set_title(f'{col} (no data)')
                continue
            ax.hist(values, bins=15, color='steelblue', alpha=0.7)
            ax.axvline(values.quantile(pct / 100.0), color='red', lw=1.5,
                       label=f'P{pct}')
            ax.set_title(col)
            ax.set_xlabel('degrees')
            ax.legend()
        plt.tight_layout()
        plt.show()

## Shoulder drift & wrist swing

These are the form signals the FSM consumes for swing / drift errors.
Values are normalised by torso length so they're comparable across subjects
and camera distances.

In [ ]:
if df.empty:
    print('Skipping — no data.')
else:
    form_cols = ['shoulder_drift_norm', 'wrist_swing_norm']
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, col in zip(axes, form_cols):
        values = df[col].dropna()
        if values.empty:
            ax.set_title(f'{col} (no data)')
            continue
        ax.hist(values, bins=15, color='darkorange', alpha=0.7)
        ax.set_title(col)
        ax.set_xlabel('ratio of torso length')
    plt.tight_layout()
    plt.show()

## Per-subject sanity check

If one subject's reps dominate the good-rep distribution the derived
thresholds are effectively fitted to them. This cell counts rows per
subject so you can spot that asymmetry before it bites the FSM.

In [ ]:
if df.empty:
    print('Skipping — no data.')
else:
    display(df.groupby('subject_id')['rep_idx'].count().rename('rows'))